# 📑 Notebook 09 — Segmentation Evaluation & PhD-Level Results
## FedMedSeg Phase 2: Final Evaluation Report

---

This notebook generates **all research-grade results** needed for your paper.

### Outputs generated:
| Output | Paper Section |
|--------|---------------|
| Dice / IoU / Pixel Accuracy (mean ± std) | Results chapter table |
| Loss & Metric curves (PDF) | Results figures |
| 20 side-by-side prediction comparisons | Appendix / Figures |
| Pixel-level confusion matrix | Evaluation chapter |
| Per-class performance breakdown | Discussion chapter |
| Failure case analysis | Limitations section |
| Complete evaluation JSON report | Supplementary material |

---
> **Prerequisite:** Run Notebooks 07 and 08 first.

In [1]:
import sys, json
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.notebook import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from segmentation.model_unet   import MobileNetV2UNet
from segmentation.metrics      import compute_all_metrics
from segmentation.dataset_rsna import RSNAPneumoniaDataset
import torchvision.transforms as T
from torch.utils.data import DataLoader

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RSNA_DIR    = PROJECT_ROOT / 'data' / 'rsna_pneumonia'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'segmentation'
WEIGHTS     = RESULTS_DIR / 'best_model_weights.pth'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

assert WEIGHTS.exists(), f'Model weights not found: {WEIGHTS}\nRun Notebook 08 first.'
print(f'Device:  {DEVICE}')
print(f'Weights: {WEIGHTS}')

Device:  cuda
Weights: c:\Users\Vivek Singh Chauhan\Desktop\shubham\FedMedSeg\results\segmentation\best_model_weights.pth


## 🔁 Part 1 — Load Best Model

In [2]:
model = MobileNetV2UNet(pretrained=False, freeze_encoder=False).to(DEVICE)
model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
model.eval()
print('✓ Best model loaded successfully!')

# Load training config
with open(RESULTS_DIR / 'training_config.json') as f:
    config = json.load(f)
print(f'\nTraining Config:')
print(f'  Epochs:      {config["num_epochs"]}')
print(f'  Batch size:  {config["batch_size"]}')
print(f'  LR Phase A:  {config["lr_phase_a"]}')
print(f'  LR Phase B:  {config["lr_phase_b"]}')
print(f'  Random Seed: {config["random_seed"]}')

✓ Best model loaded successfully!

Training Config:
  Epochs:      30
  Batch size:  16
  LR Phase A:  0.0001
  LR Phase B:  1e-05
  Random Seed: 42


In [3]:
# Load validation dataset
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_dataset = RSNAPneumoniaDataset(
    rsna_root     = str(RSNA_DIR),
    subset_csv    = str(RSNA_DIR / 'subset' / 'val_subset.csv'),
    img_transform = transform,
    augment       = False,
)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
print(f'Validation set: {len(val_dataset)} images')

[RSNAPneumoniaDataset] Loaded 1000 patients
  Pneumonia: 500
  Normal:    500
Validation set: 1000 images


## 📐 Part 2 — Full Test-Set Evaluation (Mean ± Std)

We compute metrics across **every batch** of the validation set and report **mean ± standard deviation** — this is the PhD-level reporting standard.

In [4]:
all_dice, all_iou, all_pix = [], [], []
all_preds_flat, all_masks_flat = [], []

with torch.no_grad():
    for images, masks in tqdm(val_loader, desc='Evaluating'):
        images = images.to(DEVICE)
        masks  = masks.to(DEVICE)
        preds  = model(images)

        m = compute_all_metrics(preds, masks, threshold=0.5)
        all_dice.append(m['dice'])
        all_iou.append(m['iou'])
        all_pix.append(m['pixel_acc'])

        # For confusion matrix
        pred_bin = (preds > 0.5).float().cpu().numpy().flatten()
        mask_bin = masks.cpu().numpy().flatten()
        all_preds_flat.extend(pred_bin)
        all_masks_flat.extend(mask_bin)

results = {
    'dice':      {'mean': np.mean(all_dice), 'std': np.std(all_dice)},
    'iou':       {'mean': np.mean(all_iou),  'std': np.std(all_iou)},
    'pixel_acc': {'mean': np.mean(all_pix),  'std': np.std(all_pix)},
}

print('=' * 55)
print('  FINAL TEST SET EVALUATION RESULTS')
print('  (PhD-Level Report Format: Mean ± Std)')
print('=' * 55)
print(f'  Dice Coefficient:  {results["dice"]["mean"]:.4f} ± {results["dice"]["std"]:.4f}')
print(f'  Mean IoU (Jaccard): {results["iou"]["mean"]:.4f} ± {results["iou"]["std"]:.4f}')
print(f'  Pixel Accuracy:    {results["pixel_acc"]["mean"]:.4f} ± {results["pixel_acc"]["std"]:.4f}')
print('=' * 55)

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [ ]:
# Save as publication-ready LaTeX table
latex_table = f"""
% LaTeX Table — Copy directly into your paper
\\begin{{table}}[h]
\\centering
\\caption{{Segmentation Performance of MobileNetV2-UNet on RSNA Test Set}}
\\begin{{tabular}}{{lcc}}
\\hline
\\textbf{{Metric}} & \\textbf{{Mean}} & \\textbf{{Std Dev}} \\\\
\\hline
Dice Coefficient & {results['dice']['mean']:.4f} & {results['dice']['std']:.4f} \\\\
Mean IoU (Jaccard) & {results['iou']['mean']:.4f} & {results['iou']['std']:.4f} \\\\
Pixel Accuracy & {results['pixel_acc']['mean']:.4f} & {results['pixel_acc']['std']:.4f} \\\\
\\hline
\\end{{tabular}}
\\label{{tab:seg_results}}
\\end{{table}}
"""

table_path = RESULTS_DIR / 'results_latex_table.tex'
with open(table_path, 'w') as f:
    f.write(latex_table)

print('LaTeX Table (copy into your paper):')
print(latex_table)

## 🟦 Part 3 — Pixel-Level Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

y_true = np.array(all_masks_flat).astype(int)
y_pred = np.array(all_preds_flat).astype(int)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=== Pixel-Level Confusion Matrix ===')
print(f'  True Positives  (TP): {tp:>12,}  ← Pneumonia pixels correctly detected')
print(f'  True Negatives  (TN): {tn:>12,}  ← Healthy pixels correctly ignored')
print(f'  False Positives (FP): {fp:>12,}  ← Healthy pixels wrongly marked')
print(f'  False Negatives (FN): {fn:>12,}  ← Missed pneumonia pixels')

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Healthy (0)', 'Pneumonia (1)']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Pixel-Level Confusion Matrix\nMobileNetV2-UNet on RSNA Test Set',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'pixel_confusion_matrix.pdf', format='pdf', bbox_inches='tight')
plt.savefig(RESULTS_DIR / 'pixel_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved pixel_confusion_matrix.pdf')

print('\nFull Classification Report:')
print(classification_report(y_true, y_pred, target_names=['Healthy', 'Pneumonia']))

## 📈 Part 4 — Training History Curves (Publication-Ready)

In [ ]:
from IPython.display import Image as IPImage, display

for fname, title in [('loss_curves.png', 'Loss Curves'),
                      ('dice_iou_curves.png', 'Dice & IoU Curves')]:
    print(f'=== {title} ===')
    display(IPImage(filename=str(RESULTS_DIR / fname), width=900))

## 🖼️ Part 5 — Prediction Gallery (20 Samples)

Each row shows: **Original X-ray | Ground Truth Mask | Predicted Mask | Overlay**

In [ ]:
from IPython.display import Image as IPImage, display
samples_dir = RESULTS_DIR / 'prediction_samples'
sample_files = sorted(samples_dir.glob('sample_*.png'))

print(f'Displaying all {len(sample_files)} prediction samples:')
for f in sample_files[:10]:  # Show first 10 inline
    display(IPImage(filename=str(f), width=800))

## 💀 Part 6 — Failure Case Analysis (Worst Predictions)

These are the images where the model performed worst. Identifying failure cases is a key requirement for a PhD-level paper — it shows you understand the limitations of your model.

In [ ]:
# Find the worst-performing samples (lowest Dice per image)
from segmentation.metrics import dice_coefficient
import torch.nn.functional as F

per_image_dice = []
all_images_list, all_masks_list, all_preds_list = [], [], []

with torch.no_grad():
    for images, masks in tqdm(val_loader, desc='Computing per-image Dice'):
        images_gpu = images.to(DEVICE)
        preds      = model(images_gpu).cpu()

        for i in range(images.size(0)):
            d = dice_coefficient(preds[i:i+1], masks[i:i+1])
            per_image_dice.append(d)
            all_images_list.append(images[i])
            all_masks_list.append(masks[i])
            all_preds_list.append(preds[i])

# Sort by Dice (ascending) to get worst predictions
sorted_indices = np.argsort(per_image_dice)
worst_indices  = sorted_indices[:4]   # Bottom 4

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
fig.suptitle('Failure Case Analysis — 4 Worst Predictions', fontsize=15, fontweight='bold')

for row, idx in enumerate(worst_indices):
    img = all_images_list[idx].numpy().transpose(1, 2, 0)
    img = (img * np.array(IMAGENET_STD)) + np.array(IMAGENET_MEAN)
    img = np.clip(img, 0, 1)

    gt_mask   = all_masks_list[idx][0].numpy()
    pred_mask = (all_preds_list[idx][0].numpy() > 0.5).astype(float)
    prob_mask = all_preds_list[idx][0].numpy()  # Raw probability map

    axes[row, 0].imshow(img);              axes[row, 0].set_title(f'X-ray (Dice={per_image_dice[idx]:.3f})')
    axes[row, 1].imshow(gt_mask, cmap='gray'); axes[row, 1].set_title('Ground Truth')
    axes[row, 2].imshow(pred_mask, cmap='gray'); axes[row, 2].set_title('Prediction (thresh=0.5)')
    axes[row, 3].imshow(prob_mask, cmap='jet', vmin=0, vmax=1); axes[row, 3].set_title('Probability Heatmap')

    for ax in axes[row]: ax.axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'failure_cases.pdf', format='pdf', bbox_inches='tight')
plt.savefig(RESULTS_DIR / 'failure_cases.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Saved failure_cases.pdf')

## 💾 Part 7 — Export Full Evaluation Report (JSON)

In [ ]:
from datetime import datetime

full_report = {
    'model':              'MobileNetV2-UNet',
    'dataset':            'RSNA Pneumonia Detection Challenge',
    'subset_size':        5000,
    'val_set_size':       len(val_dataset),
    'evaluation_timestamp': datetime.now().isoformat(),
    'metrics': {
        'dice_coefficient': {
            'mean': round(results['dice']['mean'], 4),
            'std':  round(results['dice']['std'], 4),
            'interpretation': 'Overlap between predicted and true mask'
        },
        'mean_iou': {
            'mean': round(results['iou']['mean'], 4),
            'std':  round(results['iou']['std'], 4),
            'interpretation': 'Jaccard Index — stricter than Dice'
        },
        'pixel_accuracy': {
            'mean': round(results['pixel_acc']['mean'], 4),
            'std':  round(results['pixel_acc']['std'], 4),
            'interpretation': 'Fraction of correctly classified pixels'
        },
    },
    'pixel_confusion_matrix': {
        'TP': int(tp), 'TN': int(tn),
        'FP': int(fp), 'FN': int(fn),
    },
    'loss_function':  'Dice-BCE Hybrid Loss',
    'optimizer':      'Adam (lr=1e-4 → 1e-5)',
    'scheduler':      'ReduceLROnPlateau (patience=5, factor=0.5)',
    'architecture':   'MobileNetV2 Encoder + Custom U-Net Decoder with skip connections',
    'training_config': config,
}

report_path = RESULTS_DIR / 'model_evaluation_report.json'
with open(report_path, 'w') as f:
    json.dump(full_report, f, indent=2)

print('✓ Full evaluation report saved to:')
print(f'  {report_path}')
print('\nReport preview:')
print(json.dumps(full_report['metrics'], indent=2))

## ✅ Final Summary — All Generated Outputs

| File | Use in Paper |
|------|--------------|
| `training_logs.csv` | Results table (all epochs) |
| `loss_curves.pdf` | Figure: Loss during training |
| `dice_iou_curves.pdf` | Figure: Metric progression |
| `pixel_confusion_matrix.pdf` | Figure: Evaluation matrix |
| `failure_cases.pdf` | Figure: Model limitations |
| `results_latex_table.tex` | Table: Direct copy into LaTeX |
| `model_evaluation_report.json` | Supplementary material |
| `prediction_samples/*.png` | Appendix: Visual evidence |
| `prediction_overlay/*.png` | Hero figures for paper |
| `best_model_weights.pth` | Reproducibility / release |
| `training_config.json` | Reproducibility section |

**You now have everything needed to write the Results and Evaluation chapters of your PhD-level research paper! 🎓**